In [6]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd

In [7]:
movies = pd.read_csv("/Users/sophie/Dev/movie-recommender-system/data/movies_cleaned_up.csv")

In [8]:
movies

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811
...,...,...,...,...,...,...,...,...,...
9980,10196,The Last Airbender,"Action,Adventure,Fantasy",en,"The story follows the adventures of Aang, a yo...",98.322,2010-06-30,4.7,3347
9981,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",en,The sharks take bite out of the East Coast whe...,12.490,2015-07-22,4.7,417
9982,13995,Captain America,"Action,Science Fiction,War",en,"During World War II, a brave, patriotic Americ...",18.333,1990-12-14,4.6,332
9983,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",en,A man named Farmer sets out to rescue his kidn...,15.159,2007-11-29,4.7,668


In [9]:
# create combined text for title, genre, and overview fields
movies["combined_text"] = (
    #"Title: " + movies["title"].fillna("") + ". " +
    "Genre: " + movies["genre"].fillna("") + ". " +
    "Overview: " + movies["overview"].fillna("")
)
movies["combined_text"]

0       Genre: Drama,Crime. Overview: Framed in the 19...
1       Genre: Comedy,Drama,Romance. Overview: Raj is ...
2       Genre: Drama,Crime. Overview: Spanning the yea...
3       Genre: Drama,History,War. Overview: The true s...
4       Genre: Drama,Crime. Overview: In the continuin...
                              ...                        
9980    Genre: Action,Adventure,Fantasy. Overview: The...
9981    Genre: Action,TV Movie,Science Fiction,Comedy,...
9982    Genre: Action,Science Fiction,War. Overview: D...
9983    Genre: Adventure,Fantasy,Action,Drama. Overvie...
9984    Genre: Thriller,Action,Crime. Overview: Seekin...
Name: combined_text, Length: 9985, dtype: str

In [10]:
# get embeddings - using hf transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    movies["combined_text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [65]:
embeddings

array([[-0.01507484, -0.05933658, -0.06913045, ...,  0.04426529,
         0.08803046, -0.05934433],
       [-0.0237387 ,  0.02333198, -0.04900789, ...,  0.02851259,
         0.00259286, -0.01558666],
       [-0.07220379,  0.03323361, -0.06877995, ...,  0.01351652,
         0.08967282, -0.05792296],
       ...,
       [-0.05933784, -0.0218488 , -0.04173438, ..., -0.02284305,
        -0.04142766, -0.00319057],
       [-0.08427592, -0.03778565, -0.02804293, ..., -0.02913866,
         0.01068622,  0.01664642],
       [-0.00747918,  0.01418289, -0.06008193, ...,  0.01504259,
         0.05038825, -0.00843891]], shape=(9985, 384), dtype=float32)

In [11]:
print(embeddings.shape)
print(movies.shape)

(9985, 384)
(9985, 10)


In [20]:
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

def recommend_movies(movie_id, movies, embeddings, top_n=10):
    match = movies[movies["id"] == movie_id] # find the movie row(s) that matches the user input
   
    if match.empty:
        return None
    
    idx = match.index[0] # gets the row of the movie in movies df
    query_vector = embeddings[idx].reshape(1, -1) # gets the movie embedding vector and changes shape for cosine similairty to work with

    similarities = cosine_similarity(query_vector, embeddings)[0] # compares with all the movie's embeddings and gets the first row from result

    top_indices = np.argsort(similarities)[::-1] # sorts movies by similarity scores
    top_indices = [i for i in top_indices if i != idx][:top_n] # returns top n movies (besides itself)

    return movies.iloc[top_indices][["id", "title", "genre", "overview"]].assign(
        similarity=similarities[top_indices]
    )

In [21]:
recommend_movies(3558, movies, embeddings)

,id,title,genre,overview,similarity
2186,581734,Nomadland,Drama,A woman in her sixties embarks on a journey th...,0.614804
441,122906,About Time,"Drama,Romance,Fantasy",The night after another unsatisfactory New Yea...,0.584199
1447,12169,The First Day of the Rest of Your Life,Drama,A sprawling drama centered on five key days in...,0.581194
1964,590,The Hours,Drama,"""The Hours"" is the story of three women search...",0.577275
884,4960,"Synecdoche, New York",Drama,"A theater director struggles with his work, an...",0.576784
1553,18917,Alice,"Animation,Fantasy,Adventure",A quiet young English girl named Alice finds h...,0.575893
450,946,Letter from an Unknown Woman,"Drama,Romance",A pianist about to flee from a duel receives a...,0.575026
6074,1989,My Blueberry Nights,"Drama,Romance",Elizabeth has just been through a particularly...,0.566208
9075,653601,Horse Girl,Drama,A socially awkward woman with a fondness for a...,0.563820
1299,10683,Happiness,"Comedy,Drama",The lives of many individuals connected by the...,0.562612


In [22]:
recommend_movies(9799, movies, embeddings)

,id,title,genre,overview,similarity
2220,385128,F9,"Action,Crime,Thriller",Dominic Toretto and his crew battle the most s...,0.773225
2248,51497,Fast Five,"Action,Thriller,Crime",Former cop Brian O'Conner partners with ex-con...,0.615781
4763,13804,Fast & Furious,"Action,Crime,Drama,Thriller","When a crime brings them back to L.A., fugitiv...",0.596247
4102,82992,Fast & Furious 6,"Action,Thriller,Crime",Hobbs has Dominic and Brian reassemble their c...,0.504916
8137,324542,Sleepless,"Action,Crime,Thriller",Undercover Las Vegas police officer Vincent Do...,0.492678
2272,168259,Furious 7,"Action,Thriller,Crime,Adventure",Deckard Shaw seeks revenge against Dominic Tor...,0.479488
9836,325358,Superfast!,"Comedy,Action",Undercover cop Lucas White joins Vin Serento's...,0.467370
5820,584,2 Fast 2 Furious,"Action,Crime,Thriller",It's a major double-cross when former police o...,0.460516
912,586863,Les Misérables,"Crime,Drama,Thriller","Inspired by the 2005 riots in Paris, Stéphane,...",0.446549
6282,136797,Need for Speed,"Action,Crime,Drama,Thriller",The film revolves around a local street-racer ...,0.434770


In [24]:
recommend_movies(68726, movies, embeddings)

,id,title,genre,overview,similarity
7001,9457,Deep Rising,"Adventure,Action,Horror,Science Fiction",A group of heavily armed hijackers board a lux...,0.714092
2888,8619,Master and Commander: The Far Side of the World,"Adventure,Drama,War",After an abrupt and violent encounter with a F...,0.709789
5031,152747,All Is Lost,"Action,Adventure,Drama","During a solo voyage in the Indian Ocean, a ve...",0.678037
1738,16320,Serenity,"Science Fiction,Action,Adventure,Thriller",When the renegade crew of Serenity agrees to h...,0.677364
1289,19995,Avatar,"Action,Adventure,Fantasy,Science Fiction","In the 22nd century, a paraplegic Marine is di...",0.669986
6348,507441,Sea Fever,"Science Fiction,Horror,Thriller",The crew of a West of Ireland trawler—marooned...,0.668417
8810,523931,Megalodon,"Action,Science Fiction,Horror,TV Movie",A military vessel on the search for an unident...,0.667509
7971,14372,Leviathan,"Adventure,Horror,Action,Thriller,Science Fiction",Underwater deep-sea miners encounter a Soviet ...,0.657493
9344,354282,The Osiris Child,Science Fiction,Set in the future in a time of interplanetary ...,0.650244
7282,597890,Voyagers,"Science Fiction,Thriller","With the future of the human race at stake, a ...",0.649336


In [61]:
import numpy as np

np.save("movie_embeddings.npy", embeddings)
movies.to_csv("movies_processed.csv", index = False)